In [1]:
from pymongo import MongoClient
import pandas as pd

try: 
    client = MongoClient("localhost", 5001)
    print("Connected successfully!!!") 
except:
    print("Could not connect to MongoDB")

#project_id = "6543b8ee4180527babd20c3a" # Anna's Paper
project_id = "656a440644dec9f71f2dee44" 

db1 = client["flask_db"]
annotations = db1.annotation
activity = db1.activity

db2 = client["dataset_db"]
data = db2.data
label = db2.label


Connected successfully!!!


/tmp/ipykernel_1993545/1321177514.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
if True:
  data.delete_many({})

In [3]:
query = {}

cursor = annotations.find(query)

annotation_df = pd.DataFrame(list(cursor))

In [4]:
annotation_df

,_id,annotatorEmail,6543b8ee4180527babd20c3a,656a440644dec9f71f2dee44,640e22cae918523bcee8ca5e
0,65e7e06e28a34165fa18a177,rossvolkov@gmail.com,"{'filledArray': [['14'], ['14'], ['14'], ['14'...","{'filledArray': [['3'], ['3'], ['3'], ['3'], [...",NaN
1,65e8dc9928a34165fa193caf,sam,"{'filledArray': [[0], [0], [0], [0], [0], [0],...","{'filledArray': [[0], [0], [0], [0], [0], [0],...",NaN
2,661422a4432ad9341e4dc8c7,lee03533@umn.edu,"{'filledArray': [['14'], ['14'], ['14'], ['14'...","{'filledArray': [['3'], ['3'], ['3'], ['3'], [...",NaN
3,65eb0e7b28a34165fa1a970c,wang9257@umn.edu,"{'filledArray': [['14'], ['14'], ['14'], ['14'...","{'filledArray': [[0], [0], [0], [0], [0], [0],...",NaN
4,66156e6e90a1e2da395e4f04,chau0139@umn.edu,NaN,"{'filledArray': [['3'], ['3'], ['3'], ['3'], [...","{'filledArray': [[0], [0], [0], [0], [0], [0],..."
5,6615ce7726063deea2521225,Nazar101@umn.edu,NaN,"{'filledArray': [['3'], ['3'], ['3'], ['3'], [...","{'filledArray': [[0], [0], [0], [0], [0], [0],..."


In [5]:
project_ids = annotation_df.columns[2:].to_list()

In [6]:
import datetime

CRUMBLED_THRESHOLD = 6

activity_dict = {}

for project_id in project_ids:
  query = {'project': project_id,
             "file": {"$exists": True},
             "$or": [{"message": {"$exists": True}}, {"state": {"$exists": True}}],
             'revision': {"$exists": True, "$ne": []},
             'editingLines': {"$exists": True, "$ne": []},
             }
  activity_data = activity.find(query).sort("timestamp", 1)

  count = 0
  
  edit_content = {}

  for edit in activity_data:
    if len(edit["revision"]) < CRUMBLED_THRESHOLD:
        edit_dict = {"index": count, "project_id": project_id, "file": edit["file"], "username": edit['username'], "timestamp": edit["timestamp"], "line_nums": edit["editingLines"], "revision": edit["revision"]}
        db2.data.insert_one(edit_dict)
    # if current edit is crumbled, we split it into two frames
    else:
        before, after = [], []
        for section in edit["revision"]:
            if section[0] == 0 or section[0] == -1:
                before.append(section)
            if section[0] == 0 or section[0] == 1:
                after.append(section)
        before_edit_dict = {"index": count, "project_id": project_id, "file": edit["file"], "username": edit['username'], "timestamp": edit["timestamp"], "line_nums": edit["editingLines"], "revision": before}
        after_edit_dict = {"index": count+1, "project_id": project_id, "file": edit["file"], "username": edit['username'], "timestamp": edit["timestamp"], "line_nums": edit["editingLines"], "revision": after}
        count += 2
        db2.data.insert_many([before_edit_dict, after_edit_dict])

  activity_dict[project_id] = edit_content